# alternance-extractor -- benchmark: fine-tuned Qwen2.5-1.5B vs Groq baseline

**Status: draft skeleton, not yet run.** Needs three things that don't exist yet:
1. `data/test/test.jsonl` -- hand-corrected test set (`label/select_test_set.py` then
   `label/review_server.py`), which itself needs the full Groq labelling run finished.
2. `data/test/test_candidates.jsonl` -- already doubles as the Groq baseline's predictions
   file (same run that produced `all_labelled.jsonl` also labelled these 100).
3. A trained LoRA adapter from `notebooks/kaggle_train.ipynb` at `ADAPTER_DIR`.

This notebook: runs the fine-tuned model over the 100 test postings, writes its predictions
in the same shape `label/run_labelling.py` uses, then scores both models with `eval/score.py`
and prints the field-level F1 / exact-match / JSON-validity / latency numbers the README's
claim needs.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mmattar18/alternance-extractor.git"
REPO_ROOT = Path("/kaggle/working/alternance-extractor")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import json
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

from schema.posting import parse_llm_json  # noqa: E402
from label.prompt import SYSTEM_PROMPT  # noqa: E402  -- same rules text used to label the data
from eval.score import score_files, print_report  # noqa: E402

## Config

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# Not "/kaggle/input/alternance-extractor-adapter" -- this Kaggle API version (kagglesdk,
# the OAuth-token CLI) mounts attached datasets at /kaggle/input/datasets/<owner>/<slug>/,
# not the classic /kaggle/input/<slug>/ documented everywhere. Confirmed by walking
# /kaggle/input in a diagnostic run -- the file was genuinely there, just not where every
# tutorial says to look.
ADAPTER_DIR = "/kaggle/input/datasets/mattarmario/alternance-extractor-adapter"

TEST_PATH = REPO_ROOT / "data" / "test" / "test.jsonl"
GROQ_PREDICTIONS_PATH = REPO_ROOT / "data" / "test" / "test_candidates.jsonl"
FINETUNED_PREDICTIONS_PATH = REPO_ROOT / "data" / "test" / "predictions_finetuned.jsonl"

MAX_NEW_TOKENS = 512
MAX_ATTEMPTS = 3  # matches label/groq_client.py's retry-on-invalid-JSON behavior, for a fair comparison

## Load base model + adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

## Inference

Mirrors `label/groq_client.py`'s `extract()`: on invalid JSON, feed the model its own bad
output plus the validation error and ask it to correct itself, up to `MAX_ATTEMPTS` times, so
the JSON-validity-rate comparison against Groq is apples-to-apples rather than giving one side
more chances than the other.

In [ ]:
def extract(raw_text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": raw_text},
    ]
    start = time.monotonic()
    last_response = ""
    last_error = ""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )
        content = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        last_response = content
        posting, error = parse_llm_json(content)
        if posting is not None:
            return {
                "prediction": posting.model_dump(),
                "valid": True,
                "error": None,
                "attempts": attempt,
                "latency_seconds": time.monotonic() - start,
            }
        last_error = error
        messages.append({"role": "assistant", "content": content})
        messages.append({
            "role": "user",
            "content": f"That was not valid: {error}. Return ONLY a corrected JSON object matching the schema.",
        })
    return {
        "prediction": None,
        "valid": False,
        "error": last_error,
        "attempts": MAX_ATTEMPTS,
        "latency_seconds": time.monotonic() - start,
    }

In [ ]:
test_records = [json.loads(l) for l in TEST_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"{len(test_records)} test postings to run")

FINETUNED_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with FINETUNED_PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
    for i, record in enumerate(test_records, 1):
        result = extract(record["raw_text"])
        out = {"posting_id": record["posting_id"], **result}
        f.write(json.dumps(out, ensure_ascii=False) + "\n")
        f.flush()
        if i % 10 == 0 or i == len(test_records):
            print(f"[{i}/{len(test_records)}]")

## Score both models

`GROQ_PREDICTIONS_PATH` (`test_candidates.jsonl`) already has Groq's `prediction`/`valid` for
these exact 100 postings from the original labelling run -- no separate Groq re-run needed, and
re-running would just add noise since `temperature=0` already makes it deterministic.

In [ ]:
print("=" * 30, "GROQ BASELINE", "=" * 30)
groq_result = score_files(TEST_PATH, GROQ_PREDICTIONS_PATH)
print_report(groq_result)

print()
print("=" * 30, "FINE-TUNED Qwen2.5-1.5B", "=" * 30)
finetuned_result = score_files(TEST_PATH, FINETUNED_PREDICTIONS_PATH)
print_report(finetuned_result)

## Latency comparison

Read directly off `latency_seconds` in each predictions file -- no cost numbers hardcoded here
deliberately, since Groq's per-token pricing can change and a stale hardcoded figure would be
worse than none. Pull current pricing at analysis time and multiply by measured token counts.

In [ ]:
def latency_stats(path):
    lats = [json.loads(l)["latency_seconds"] for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    lats.sort()
    n = len(lats)
    return {"n": n, "median": lats[n // 2], "mean": sum(lats) / n, "p95": lats[int(n * 0.95)]}

print("Groq baseline latency (s):     ", latency_stats(GROQ_PREDICTIONS_PATH))
print("Fine-tuned model latency (s):  ", latency_stats(FINETUNED_PREDICTIONS_PATH))

## Next step

Copy the printed numbers (field-level F1, exact-match rate, JSON-validity rate, latency) into
the README's status section, replacing the "Numbers below are not measured" placeholder --
that's the actual writeup the README says is still pending.